#Initializations

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import StringType

#Read from Bronze Layer Tables

In [0]:
df= spark.table("workspace.bronze.crm_cust_info")

#Transformations

##Renaming Columns 

In [0]:
Rename_col= {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "firstname",
    "cst_lastname": "lastname",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "create_date"
}

for old_name , new_name in Rename_col.items():
    df=df.withColumnRenamed(old_name,new_name)

##Trimming 


In [0]:

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))


##Normalizations

In [0]:
df=(
    df
    .withColumn("marital_status",
                when(upper(col("marital_status")) =="M","Married")
                .when(upper(col("marital_status")) =="S","Single")
                .otherwise("n/a"))
    
    .withColumn("gender",
                when(upper(col("gender")) =="M","Male")
                .when(upper(col("gender")) =="F","Female")
                .otherwise("n/a"))
    
)

#Writing To Silver Table

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.crm_customers")
